In [4]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from scipy import stats
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

#=====================================
# STEP 1: IMPORT DATA
#=====================================
print("="*60)
print("LOADING DATA FROM STATA FILE")
print("="*60)

# Import the Stata file
df = pd.read_stata('/content/randhrs1992_2020v2.dta')

# Display basic information about the dataset
print(f"\nDataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows of the data:")
print(df.head())

print("\nColumn names in the dataset:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nBasic statistics:")
print(df.describe())

print("\nMissing values per column:")
print(df.isnull().sum())

#=====================================
# MAIN CLASS DEFINITION
#=====================================

class ArellanoBohhommeQuantileDecomposition:
    """
    Arellano and Bonhomme (2016) Grouped Fixed Effects for Quantile Regression
    Matching Stata command: rqdeco lnhitot hchild rmstat raracem ragey_b raeduc
                           rahispan ravetrn rabplace i.round, by(nokids) quantile(0.5)
    """

    def __init__(self, data, n_groups=10):
        """
        Parameters:
        -----------
        data : pandas DataFrame with columns:
            - lnhitot: log household income total (dependent variable)
            - hchild: has children dummy
            - rmstat: marital status
            - raracem: race/ethnicity categories
            - ragey_b: age at baseline
            - raeduc: education level
            - rahispan: Hispanic indicator
            - ravetrn: veteran status
            - rabplace: birthplace
            - round: survey round/wave
            - nokids: number of kids (treatment variable for decomposition)
            - hhidpn or id: household/person identifier
        n_groups : int
            Number of groups for grouped fixed effects
        """
        self.data = data.copy()
        self.n_groups = n_groups

        # Define variables based on your specification
        self.dep_var = 'lnhhincome' # Corrected variable name
        self.treatment_var = 'nokids'  # by(nokids) in rqdeco

        # Control variables
        self.control_vars = [
            'hchild', 'rmstat', 'raracem', 'ragey_b',
            'raeduc', 'rahispan', 'ravetrn', 'rabplace'
        ]

        # Check which variables exist in the data
        print("\nChecking for required variables...")
        missing_vars = []
        for var in [self.dep_var, self.treatment_var] + self.control_vars:
            if var not in self.data.columns:
                missing_vars.append(var)
                print(f"  ⚠ Missing: {var}")
            else:
                print(f"  ✓ Found: {var}")

        if missing_vars:
            print(f"\nWarning: Missing variables {missing_vars}")
            # Remove missing variables from control_vars
            self.control_vars = [v for v in self.control_vars if v in self.data.columns]

        # Identify ID variable (try common names)
        if 'hhidpn' in self.data.columns:
            self.id_var = 'hhidpn'
        elif 'id' in self.data.columns:
            self.id_var = 'id'
        elif 'pid' in self.data.columns:
            self.id_var = 'pid'
        else:
            # Create ID if not exists
            print("\nCreating ID variable...")
            self.data['id'] = range(len(self.data))
            self.id_var = 'id'

        print(f"\nUsing '{self.id_var}' as ID variable")

        self.time_var = 'round' if 'round' in self.data.columns else None

        if self.time_var:
            # Create dummy variables for round (i.round in Stata)
            self.round_dummies = pd.get_dummies(self.data['round'], prefix='round', drop_first=True)
            self.round_dummy_cols = self.round_dummies.columns.tolist()

            # Add round dummies to data
            self.data = pd.concat([self.data, self.round_dummies], axis=1)
        else:
            print("\nWarning: 'round' variable not found - proceeding without time effects")
            self.round_dummy_cols = []

        # Check for missing values
        self._handle_missing_values()

        # Convert categorical control variables to dummies
        self._handle_categorical_variables()


    def _handle_missing_values(self):
        """Handle missing values in the dataset"""
        # Report missing values
        vars_to_check = [v for v in [self.dep_var, self.treatment_var] + self.control_vars
                        if v in self.data.columns]

        missing_report = self.data[vars_to_check].isnull().sum()
        if missing_report.sum() > 0:
            print("\nMissing values detected:")
            print(missing_report[missing_report > 0])
            print(f"\nDropping {missing_report.sum()} observations with missing values...")

        # Drop missing values
        self.data = self.data.dropna(subset=vars_to_check)
        print(f"Final sample size: {len(self.data)} observations")

    def _handle_categorical_variables(self):
        """Convert categorical control variables to dummy variables."""
        categorical_vars = self.data[self.control_vars].select_dtypes(include='category').columns.tolist()
        if categorical_vars:
            print(f"\nConverting categorical variables to dummies: {categorical_vars}")
            self.data = pd.get_dummies(self.data, columns=categorical_vars, drop_first=True)

            # Update control_vars to include dummy variable names
            original_control_vars = self.control_vars.copy()
            self.control_vars = [var for var in self.control_vars if var not in categorical_vars] # remove original categorical vars
            for cat_var in categorical_vars:
                 # Add new dummy column names
                 self.control_vars.extend([col for col in self.data.columns if col.startswith(f'{cat_var}_')])

            print(f"Updated control variables: {self.control_vars}")


    def decomposition_setup(self):
        """
        Set up for quantile decomposition by nokids groups
        Following rqdeco's by(nokids) specification
        """
        # Create treatment groups based on nokids
        if self.treatment_var in self.data.columns:
            self.data['has_kids'] = (self.data[self.treatment_var] > 0).astype(int)

            # Split data by treatment status
            self.data_nokids = self.data[self.data['has_kids'] == 0].copy()
            self.data_kids = self.data[self.data['has_kids'] == 1].copy()

            print(f"\nSample sizes by kids status:")
            print(f"  No kids (control): {len(self.data_nokids)}")
            print(f"  Has kids (treatment): {len(self.data_kids)}")

            # For continuous nokids, create groups
            if self.data[self.treatment_var].nunique() > 5:
                # Create categories for number of kids
                self.data['nokids_cat'] = pd.cut(self.data[self.treatment_var],
                                                bins=[-0.5, 0.5, 1.5, 2.5, 3.5, 100],
                                                labels=['0', '1', '2', '3', '4+'])

                print(f"\nDistribution of number of kids:")
                print(self.data['nokids_cat'].value_counts().sort_index())

    def estimate_initial_quantile_fe(self, tau=0.5, data_subset=None):
        """
        Step 1: Initial quantile regression to get preliminary fixed effects
        """
        if data_subset is None:
            data_subset = self.data

        # Prepare variables (controls + round dummies)
        X_vars = [var for var in self.control_vars + self.round_dummy_cols if var in data_subset.columns]
        X = data_subset[X_vars]
        y = data_subset[self.dep_var]

        # Convert boolean columns to int
        for col in X.columns:
            if X[col].dtype == 'bool':
                X[col] = X[col].astype(int)


        # Add constant
        X = sm.add_constant(X)

        # Debug: Check data types before regression
        print("\nData types of X before initial QuantReg:")
        print(X.dtypes)

        # Run initial quantile regression
        initial_qreg = QuantReg(y, X).fit(q=tau)

        # Calculate residuals
        residuals = y - initial_qreg.predict(X)

        # Estimate individual-specific effects from residuals
        if self.id_var in data_subset.columns:
            fe_initial = residuals.groupby(data_subset[self.id_var]).mean()
        else:
            fe_initial = residuals

        return initial_qreg, fe_initial

    def create_groups_AB2016(self, fe_estimates, data_subset=None):
        """
        Step 2: Create groups following Arellano-Bonhomme (2016)
        """
        if data_subset is None:
            data_subset = self.data

        # Method 1: K-means clustering on FE
        fe_array = fe_estimates.values.reshape(-1, 1)
        kmeans = KMeans(n_clusters=self.n_groups, random_state=42, n_init=10)
        groups = kmeans.fit_predict(fe_array)

        # Create group mapping
        if isinstance(fe_estimates.index, pd.Index):
            group_mapping = dict(zip(fe_estimates.index, groups))
        else:
            group_mapping = dict(enumerate(groups))

        return group_mapping

    def estimate_grouped_quantile_regression(self, tau=0.5):
        """
        Step 3: Main estimation with grouped fixed effects
        """
        results = {}

        print(f"\n{'='*60}")
        print(f"Arellano-Bonhomme Grouped FE Quantile Regression (τ={tau})")
        print('='*60)

        # Estimate for full sample
        print("\n1. Full Sample Estimation:")
        print("-" * 40)

        # Get initial FE
        initial_model, fe_initial = self.estimate_initial_quantile_fe(tau=tau)

        # Create groups
        group_mapping = self.create_groups_AB2016(fe_initial)
        self.data['ab_group'] = self.data[self.id_var].map(group_mapping)

        # Create group-round interactions if round exists
        if self.time_var:
            self.data['group_round'] = (self.data['ab_group'].astype(str) + '_' +
                                       self.data['round'].astype(str))
            group_round_dummies = pd.get_dummies(self.data['group_round'],
                                                prefix='gr', drop_first=True)
        else:
            # Just use group dummies if no time variable
            group_round_dummies = pd.get_dummies(self.data['ab_group'],
                                                prefix='group', drop_first=True)

        # Prepare final regression
        X_vars = [var for var in self.control_vars + list(group_round_dummies.columns) if var in self.data.columns]
        X = self.data[X_vars]
        y = self.data[self.dep_var]

        # Convert boolean columns to int
        for col in X.columns:
            if X[col].dtype == 'bool':
                X[col] = X[col].astype(int)

        # Add constant
        X = sm.add_constant(X)

        # Debug: Check data types before regression
        print("\nData types of X before grouped QuantReg:")
        print(X.dtypes)


        # Run grouped FE quantile regression
        grouped_qreg = QuantReg(y, X).fit(q=tau)

        # Store results
        results['full_sample'] = {
            'model': grouped_qreg,
            'coefficients': grouped_qreg.params[[col for col in self.control_vars if col in grouped_qreg.params]],
            'std_errors': grouped_qreg.bse[[col for col in self.control_vars if col in grouped_qreg.bse]]
        }

        # Print results
        print("\nCoefficients (with grouped FE):")
        for var in self.control_vars:
            coef = grouped_qreg.params[var] if var in grouped_qreg.params else np.nan
            se = grouped_qreg.bse[var] if var in grouped_qreg.bse else np.nan
            print(f"  {var:15s}: {coef:8.4f} (SE: {se:.4f})")

        # 2. Decomposition by nokids groups
        if self.treatment_var in self.data.columns:
            print("\n2. Decomposition by Number of Kids:")
            print("-" * 40)

            self.decomposition_setup()

            # Estimate for each group
            for group_name, group_data in [('No Kids', self.data_nokids),
                                           ('Has Kids', self.data_kids)]:
                if len(group_data) > 50:  # Need minimum sample size
                    print(f"\n{group_name} Group (n={len(group_data)}):")

                    try:
                        # Get initial FE for this group
                        group_model, group_fe = self.estimate_initial_quantile_fe(
                            tau=tau, data_subset=group_data
                        )

                        # Create groups
                        group_mapping = self.create_groups_AB2016(group_fe, group_data)
                        group_data['ab_group'] = group_data[self.id_var].map(group_mapping)

                        # Create dummies
                        if self.time_var and 'round' in group_data.columns:
                            group_data['group_round'] = (group_data['ab_group'].astype(str) + '_' +
                                                        group_data['round'].astype(str))
                            gr_dummies = pd.get_dummies(group_data['group_round'],
                                                       prefix=f'gr_{group_name.lower().replace(" ", "_")}',
                                                       drop_first=True)
                        else:
                            gr_dummies = pd.get_dummies(group_data['ab_group'],
                                                       prefix=f'group_{group_name.lower().replace(" ", "_")}',
                                                       drop_first=True)

                        # Prepare regression
                        X_vars = [var for var in self.control_vars + list(gr_dummies.columns) if var in group_data.columns]
                        X = group_data[X_vars]
                        y = group_data[self.dep_var]

                        # Convert boolean columns to int
                        for col in X.columns:
                            if X[col].dtype == 'bool':
                                X[col] = X[col].astype(int)

                        X = sm.add_constant(X)

                        # Debug: Check data types before grouped QuantReg for subgroup
                        print(f"\nData types of X before grouped QuantReg for {group_name}:")
                        print(X.dtypes)

                        # Run regression
                        group_qreg = QuantReg(y, X).fit(q=tau)

                        # Store results
                        results[group_name.lower().replace(' ', '_')] = {
                            'model': group_qreg,
                            'coefficients': group_qreg.params[[col for col in self.control_vars if col in group_qreg.params]],
                            'std_errors': group_qreg.bse[[col for col in self.control_vars if col in group_qreg.bse]],
                            'n': len(group_data)
                        }

                        # Print coefficients
                        for var in self.control_vars:
                             if var in group_qreg.params:
                                coef = group_qreg.params[var]
                                se = group_qreg.bse[var]
                                print(f"  {var:15s}: {coef:8.4f} (SE: {se:.4f})")
                    except Exception as e:
                        print(f"  Error estimating for {group_name}: {e}")

        self.results = results
        return results

    def estimate_multiple_quantiles(self, tau_list=[0.1, 0.25, 0.5, 0.75, 0.9]):
        """
        Estimate model for multiple quantiles
        """
        all_results = {}
        decomposition_results = []

        for tau in tau_list:
            print(f"\n{'='*70}")
            print(f"ESTIMATING FOR QUANTILE τ = {tau}")
            print('='*70)

            results = self.estimate_grouped_quantile_regression(tau=tau)
            all_results[tau] = results

            # Calculate decomposition if both groups exist
            if 'no_kids' in results and 'has_kids' in results:
                # Get mean outcomes for each group at this quantile
                y_no_kids = self.data_nokids[self.dep_var].quantile(tau)
                y_has_kids = self.data_kids[self.dep_var].quantile(tau)
                total_gap = y_has_kids - y_no_kids

                # Get mean characteristics
                # Ensure only common control variables are used for mean calculation
                common_control_vars = list(set(self.control_vars) & set(self.data_nokids.columns) & set(self.data_kids.columns))
                X_no_kids = self.data_nokids[common_control_vars].mean()
                X_has_kids = self.data_kids[common_control_vars].mean()

                # Get coefficients - ensure coefficients align with common_control_vars
                beta_no_kids = results['no_kids']['coefficients'].filter(items=common_control_vars)
                beta_has_kids = results['has_kids']['coefficients'].filter(items=common_control_vars)


                # Calculate decomposition
                explained = np.sum((X_has_kids - X_no_kids) * beta_no_kids)
                unexplained = np.sum(X_has_kids * (beta_has_kids - beta_no_kids))

                decomposition_results.append({
                    'quantile': tau,
                    'total_gap': total_gap,
                    'explained': explained,
                    'unexplained': unexplained,
                    'explained_pct': 100 * explained / total_gap if total_gap != 0 and not np.isnan(total_gap) else np.nan,
                    'unexplained_pct': 100 * unexplained / total_gap if total_gap != 0 and not np.isnan(total_gap) else np.nan
                })

                print(f"\nDecomposition at τ={tau}:")
                print(f"  Total gap: {total_gap:.4f}")
                print(f"  Explained: {explained:.4f} ({100*explained/total_gap:.1f}%)" if total_gap != 0 and not np.isnan(total_gap) else "N/A")
                print(f"  Unexplained: {unexplained:.4f} ({100*unexplained/total_gap:.1f}%)" if total_gap != 0 and not np.isnan(total_gap) else "N/A")


        self.all_quantile_results = all_results
        self.decomposition_results = pd.DataFrame(decomposition_results)

        return all_results

    def plot_decomposition(self):
        """
        Create comprehensive decomposition graphs
        """
        if not hasattr(self, 'decomposition_results') or self.decomposition_results.empty:
            print("No decomposition results to plot. Run estimate_multiple_quantiles() first.")
            return

        # Drop rows with NaN in total_gap before plotting
        plot_data = self.decomposition_results.dropna(subset=['total_gap']).copy()

        if plot_data.empty:
            print("No valid decomposition results to plot after dropping NaNs.")
            return # Returning None instead of fig as we plot individually


        # Define colors
        color_explained = '#2E86AB'
        color_unexplained = '#A23B72'
        color_total = '#F18F01'

        # ========== SUBPLOT 1: Stacked Bar Chart ==========
        fig1, ax1 = plt.subplots(figsize=(8, 6))
        quantiles = plot_data['quantile']
        explained = plot_data['explained']
        unexplained = plot_data['unexplained']

        width = 0.6
        x = np.arange(len(quantiles))

        bars1 = ax1.bar(x, explained, width, label='Explained (Characteristics)',
                       color=color_explained, alpha=0.8)
        bars2 = ax1.bar(x, unexplained, width, bottom=explained,
                       label='Unexplained (Coefficients)', color=color_unexplained, alpha=0.8)

        ax1.set_xlabel('Quantile (τ)', fontsize=11)
        ax1.set_ylabel('Contribution to Income Gap', fontsize=11)
        ax1.set_title('Decomposition of Income Gap: Kids vs No Kids', fontsize=12, fontweight='bold')
        ax1.set_xticks(x)
        ax1.set_xticklabels([f'{q:.1f}' for q in quantiles])
        ax1.legend(loc='upper left')
        ax1.grid(True, alpha=0.3, axis='y')
        ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

        # Add value labels on bars
        for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
            height1 = bar1.get_height()
            height2 = bar2.get_height()
            if height1 != 0 and not np.isnan(height1):
                ax1.text(bar1.get_x() + bar1.get_width()/2., height1/2,
                        f'{height1:.3f}', ha='center', va='center', fontsize=9)
            if height2 != 0 and not np.isnan(height2):
                ax1.text(bar2.get_x() + bar2.get_width()/2., height1 + height2/2,
                        f'{height2:.3f}', ha='center', va='center', fontsize=9)
        fig1.tight_layout()
        fig1.savefig('decomposition_stacked_bar.png', dpi=300)
        print("\nDecomposition stacked bar chart saved as 'decomposition_stacked_bar.png'")
        plt.close(fig1)


        # ========== SUBPLOT 2: Line Plot with Components ==========
        fig2, ax2 = plt.subplots(figsize=(8, 6))
        ax2.plot(plot_data['quantile'], plot_data['total_gap'],
                marker='o', linewidth=2.5, markersize=8, label='Total Gap', color=color_total)
        ax2.plot(plot_data['quantile'], explained,
                marker='s', linewidth=2, markersize=7, label='Explained', color=color_explained)
        ax2.plot(plot_data['quantile'], unexplained,
                marker='^', linewidth=2, markersize=7, label='Unexplained', color=color_unexplained)

        ax2.set_xlabel('Quantile (τ)', fontsize=11)
        ax2.set_ylabel('Gap Size', fontsize=11)
        ax2.set_title('Income Gap Components Across Quantiles', fontsize=12, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        fig2.tight_layout()
        fig2.savefig('decomposition_line_plot.png', dpi=300)
        print("Decomposition line plot saved as 'decomposition_line_plot.png'")
        plt.close(fig2)


        # ========== SUBPLOT 3: Percentage Decomposition ==========
        fig3, ax3 = plt.subplots(figsize=(8, 6))
        explained_pct = plot_data['explained_pct']
        unexplained_pct = plot_data['unexplained_pct']

        width = 0.35
        x = np.arange(len(quantiles))

        bars1 = ax3.bar(x - width/2, explained_pct, width, label='Explained %',
                       color=color_explained, alpha=0.8)
        bars2 = ax3.bar(x + width/2, unexplained_pct, width, label='Unexplained %',
                       color=color_unexplained, alpha=0.8)

        ax3.set_xlabel('Quantile (τ)', fontsize=11)
        ax3.set_ylabel('Percentage of Total Gap', fontsize=11)
        ax3.set_title('Relative Contribution to Income Gap (%)', fontsize=12, fontweight='bold')
        ax3.set_xticks(x)
        ax3.set_xticklabels([f'{q:.1f}' for q in quantiles])
        ax3.legend()
        ax3.grid(True, alpha=0.3, axis='y')
        ax3.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax3.axhline(y=100, color='gray', linestyle='--', alpha=0.5)

        # Add percentage labels
        for bar in bars1:
            height = bar.get_height()
            if abs(height) > 5 and not np.isnan(height):
                ax3.text(bar.get_x() + bar.get_width()/2., height + 2,
                        f'{height:.0f}%', ha='center', va='bottom', fontsize=9)
        for bar in bars2:
            height = bar.get_height()
            if abs(height) > 5 and not np.isnan(height):
                ax3.text(bar.get_x() + bar.get_width()/2., height + 2,
                        f'{height:.0f}%', ha='center', va='bottom', fontsize=9)
        fig3.tight_layout()
        fig3.savefig('decomposition_percentage.png', dpi=300)
        print("Decomposition percentage chart saved as 'decomposition_percentage.png'")
        plt.close(fig3)


        # ========== SUBPLOT 4: Area Plot ==========
        fig4, ax4 = plt.subplots(figsize=(8, 6))
        ax4.fill_between(plot_data['quantile'], 0, explained, alpha=0.6, color=color_explained,
                        label='Explained', step='mid')
        ax4.fill_between(plot_data['quantile'], explained, explained + unexplained,
                        alpha=0.6, color=color_unexplained, label='Unexplained', step='mid')
        ax4.plot(plot_data['quantile'], plot_data['total_gap'],
                'k-', linewidth=2, label='Total Gap')

        ax4.set_xlabel('Quantile (τ)', fontsize=11)
        ax4.set_ylabel('Contribution to Gap', fontsize=11)
        ax4.set_title('Cumulative Decomposition Across Quantiles', fontsize=12, fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        fig4.tight_layout()
        fig4.savefig('decomposition_area_plot.png', dpi=300)
        print("Decomposition area plot saved as 'decomposition_area_plot.png'")
        plt.close(fig4)


        # ========== SUBPLOT 5: Income Distributions ==========
        fig5, ax5 = plt.subplots(figsize=(8, 6))
        if hasattr(self, 'data_nokids') and hasattr(self, 'data_kids'):
            # Plot kernel density estimates
            from scipy.stats import gaussian_kde

            nokids_income = self.data_nokids[self.dep_var].dropna()
            kids_income = self.data_kids[self.dep_var].dropna()

            if not nokids_income.empty and not kids_income.empty:
                kde_nokids = gaussian_kde(nokids_income)
                kde_kids = gaussian_kde(kids_income)

                x_range = np.linspace(min(nokids_income.min(), kids_income.min()),
                                     max(nokids_income.max(), kids_income.max()), 200)

                ax5.fill_between(x_range, kde_nokids(x_range), alpha=0.5,
                               color='blue', label='No Kids')
                ax5.fill_between(x_range, kde_kids(x_range), alpha=0.5,
                               color='red', label='Has Kids')

                # Add quantile lines
                for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
                    if q <= nokids_income.quantile(1) and q >= nokids_income.quantile(0):
                         ax5.axvline(nokids_income.quantile(q), color='blue',
                                   linestyle='--', alpha=0.3, linewidth=0.8)
                    if q <= kids_income.quantile(1) and q >= kids_income.quantile(0):
                         ax5.axvline(kids_income.quantile(q), color='red',
                                   linestyle='--', alpha=0.3, linewidth=0.8)


                ax5.set_xlabel('Log Household Income (lnhhincome)', fontsize=11) # Corrected label
                ax5.set_ylabel('Density', fontsize=11)
                ax5.set_title('Income Distribution by Kids Status', fontsize=12, fontweight='bold')
                ax5.legend()
                ax5.grid(True, alpha=0.3)
            else:
                 ax5.text(0.5, 0.5, "Insufficient data for distribution plot",
                         horizontalalignment='center', verticalalignment='center',
                         transform=ax5.transAxes, fontsize=10, color='gray')
                 ax5.set_title('Income Distribution by Kids Status', fontsize=12, fontweight='bold')
                 ax5.axis('off')
        fig5.tight_layout()
        fig5.savefig('income_distribution.png', dpi=300)
        print("Income distribution plot saved as 'income_distribution.png'")
        plt.close(fig5)

        # ========== SUBPLOT 6: Decomposition Summary Table ==========
        # Tables are not typically saved as image files in this way,
        # but we can represent it visually or just rely on the CSV output.
        # For now, we will not create a separate image for the table here.
        print("\nDecomposition summary table is available in 'decomposition_results.csv'")


        # Overall title is not needed for separate plots
        # fig.suptitle('Arellano-Bonhomme (2016) Quantile Decomposition Analysis\nIncome Gap: Kids vs No Kids',
        #             fontsize=14, fontweight='bold', y=1.02)

        # plt.tight_layout() # Not needed when saving individual figures

        # Save the figure (Removed as we save individual subplots)
        # plt.savefig('decomposition_analysis.png', dpi=300, bbox_inches='tight')
        # print("\nDecomposition graph saved as 'decomposition_analysis.png'")

        # plt.show() # Show individual plots if needed, but closing them to avoid clutter

        # Return None or list of figure objects if desired
        return # Returning None as plots are saved directly

    def plot_coefficient_heterogeneity(self):
        """
        Plot how coefficients vary across quantiles
        """
        if not hasattr(self, 'all_quantile_results'):
            print("Run estimate_multiple_quantiles() first")
            return

        # Prepare data for plotting
        tau_values = sorted(self.all_quantile_results.keys())

        # Select key variables to plot
        key_vars = ['hchild', 'raeduc', 'rmstat', 'rahispan'] if 'hchild' in self.control_vars else self.control_vars[:4]
        # Filter key_vars to only include those that became dummy variables if they were categorical
        key_vars_to_plot = []
        for var in key_vars:
            if var in self.data.columns and self.data[var].dtype != 'category': # Include if it was originally numeric
                 key_vars_to_plot.append(var)
            else: # Check for dummy variables if it was originally categorical
                 dummy_cols = [col for col in self.control_vars if col.startswith(f'{var}_')]
                 key_vars_to_plot.extend(dummy_cols)

        # Limit to at most 4 variables to plot
        key_vars_to_plot = key_vars_to_plot[:4]


        if not key_vars_to_plot:
            print("No suitable control variables found to plot coefficient heterogeneity.")
            return None

        # Create separate figures for each coefficient plot
        figures = {}
        for idx, var in enumerate(key_vars_to_plot):

            fig, ax = plt.subplots(figsize=(8, 6))

            # Collect coefficients across quantiles
            coefs_full = []
            coefs_nokids = []
            coefs_kids = []

            for tau in tau_values:
                # Full sample
                if 'full_sample' in self.all_quantile_results[tau]:
                    coef = self.all_quantile_results[tau]['full_sample']['coefficients'].get(var, np.nan)
                    coefs_full.append(coef)

                # No kids group
                if 'no_kids' in self.all_quantile_results[tau]:
                    coef = self.all_quantile_results[tau]['no_kids']['coefficients'].get(var, np.nan)
                    coefs_nokids.append(coef)

                # Has kids group
                if 'has_kids' in self.all_quantile_results[tau]:
                    coef = self.all_quantile_results[tau]['has_kids']['coefficients'].get(var, np.nan)
                    coefs_kids.append(coef)

            # Plot
            ax.plot(tau_values, coefs_full, 'o-', label='Full Sample', linewidth=2, markersize=7)
            if coefs_nokids and not all(np.isnan(coefs_nokids)):
                ax.plot(tau_values, coefs_nokids, 's--', label='No Kids', linewidth=1.5, markersize=6)
            if coefs_kids and not all(np.isnan(coefs_kids)):
                ax.plot(tau_values, coefs_kids, '^--', label='Has Kids', linewidth=1.5, markersize=6)

            ax.set_xlabel('Quantile (τ)', fontsize=10)
            ax.set_ylabel('Coefficient', fontsize=10)
            ax.set_title(f'Effect of {var}', fontsize=11, fontweight='bold')
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

            fig.tight_layout()
            # Create a valid filename by replacing invalid characters
            valid_var_name = var.replace('/', '_').replace('.', '_').replace(' ', '_')
            filename = f'coefficient_heterogeneity_{valid_var_name}.png'
            fig.savefig(filename, dpi=300)
            print(f"Coefficient plot for {var} saved as '{filename}'")
            plt.close(fig) # Close the figure to free up memory
            figures[var] = fig # Store figure object if needed later

        # Return the dictionary of figures, or None
        return figures


#=====================================
# MAIN EXECUTION
#=====================================

if __name__ == "__main__":

    # Initialize the Arellano-Bonhomme model with your data
    print("\n" + "="*60)
    print("INITIALIZING ARELLANO-BONHOMME (2016) MODEL")
    print("="*60)

    ab_model = ArellanoBohhommeQuantileDecomposition(data=df, n_groups=10)

    # # Find optimal number of groups using cross-validation
    # print("\n" + "="*60)
    # print("CROSS-VALIDATION FOR OPTIMAL GROUP SIZE")
    # print("="*60)

    # optimal_k, cv_scores = ab_model.cross_validate_groups(tau=0.5, k_range=range(5, 16))
    # print(f"\nOptimal number of groups: {optimal_k}")

    # # Re-initialize with optimal groups
    # ab_model.n_groups = optimal_k

    # Main estimation at median (matching your quantile(0.5))
    print("\n" + "="*60)
    print("MAIN RESULTS AT MEDIAN (τ=0.5)")
    print("="*60)

    results_median = ab_model.estimate_grouped_quantile_regression(tau=0.5)

    # Estimate for multiple quantiles
    print("\n" + "="*60)
    print("MULTIPLE QUANTILE ESTIMATION")
    print("="*60)

    all_results = ab_model.estimate_multiple_quantiles(
        tau_list=[0.1, 0.25, 0.5, 0.75, 0.9]
    )

    # Create decomposition graphs
    print("\n" + "="*60)
    print("GENERATING DECOMPOSITION GRAPHS")
    print("="*60)

    # Main decomposition plot (saves individual subplots)
    ab_model.plot_decomposition()

    # Coefficient heterogeneity plot (saves individual coefficient plots)
    ab_model.plot_coefficient_heterogeneity()

    # Summary statistics
    print("\n" + "="*60)
    print("DECOMPOSITION SUMMARY")
    print("="*60)

    if hasattr(ab_model, 'decomposition_results') and not ab_model.decomposition_results.empty:
        print("\nDecomposition Results Across Quantiles:")
        print(ab_model.decomposition_results.to_string(index=False))

        # Save results to CSV
        ab_model.decomposition_results.to_csv('decomposition_results.csv', index=False)
        print("\nResults saved to 'decomposition_results.csv'")

    # Export all results
    print("\n" + "="*60)
    print("EXPORTING COMPLETE RESULTS")
    print("="*60)

    # Save all results to pickle
    import pickle
    results_to_save = {
        'median_results': results_median,
        'all_quantiles': all_results,
        'decomposition': ab_model.decomposition_results if hasattr(ab_model, 'decomposition_results') else None,
        # 'optimal_groups': optimal_k # Removed as cross_validate_groups is not implemented
    }

    with open('arellano_bonhomme_complete_results.pkl', 'wb') as f:
        pickle.dump(results_to_save, f)

    print("\nAll results saved to:")
    print("  - arellano_bonhomme_complete_results.pkl (full results)")
    print("  - decomposition_results.csv (decomposition summary)")
    print("  - Individual decomposition plots (see above messages)")
    print("  - Individual coefficient heterogeneity plots (see above messages)")

    print("\n" + "="*60)
    print("ANALYSIS COMPLETE")
    print("="*60)

LOADING DATA FROM STATA FILE

Dataset shape: 42405 rows, 17656 columns

First 5 rows of the data:
     hhidpn  s1hhidpn          r1mstat r1mpart  s1bmonth  s1byear  s1bdate  \
0      1010       0.0       5.divorced    0.no       NaN      NaN      NaN   
1      2010       0.0        7.widowed    0.no       NaN      NaN      NaN   
2      3010    3020.0        1.married    0.no       9.0   1938.0  -7778.0   
3      3020    3010.0        1.married    0.no       1.0   1936.0  -8752.0   
4  10001010       0.0  8.never married    0.no       NaN      NaN      NaN   

      s1bflag s1cohbyr                        s1hrsamp  ... r10lbsatwlf  \
0         NaN      NaN                             NaN  ...         NaN   
1         NaN      NaN                             NaN  ...         NaN   
2  0.mo/yr ok    3.hrs  1.in samp,hrs92 resp b.1931-41  ...         6.4   
3  0.mo/yr ok    3.hrs  1.in samp,hrs92 resp b.1931-41  ...         4.2   
4         NaN      NaN                             NaN  ..